# Single-layer ablation of the Fourier calculator subspace

Part 4 of the *Manifold Features* series. Localises Goodfire's six-clock counting circuit by layer. For each `L ∈ {15, 16, 17, 18, 19, 20, 21, 22}` we train a fresh 12-dim Fourier probe basis on 2000 addition prompts at that layer's last-token residual, then ablate the corresponding 12-dim subspace via a forward hook and measure accuracy on four tasks:

- **addition** (probes' native task), **weekdays**, **months** — the three counting tasks Goodfire used.
- **country-capital** — a non-counting control. Same number of prompts; no Fourier structure expected.

Phase 2 also runs joint ablation across {L18 only, L17-19, L16-20, L15-22} to show how the residual capability scales with the number of layers cleaned simultaneously.

**Source model**: `meta-llama/Llama-3.1-8B` (gated; needs `HF_TOKEN`).

**Outputs**: `./out/multilayer_ablation/{phase1.json, phase2.json, plot.png}`.

**Runs on**: a single A100 (40 GB) in ~20 minutes. With `RECOMPUTE = False` the plot rebuilds from `../data/multilayer_ablation_phase1.json` and `../data/multilayer_ablation_phase2.json` in seconds.


In [ ]:
# Cell 1 — setup
import os, sys, json, time, subprocess
from pathlib import Path

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import numpy as np
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from torch.nn.functional import softmax
    from sklearn.linear_model import Ridge
    import matplotlib.pyplot as plt
except ImportError:
    _pip('torch', 'transformers', 'numpy', 'scikit-learn', 'matplotlib', 'huggingface_hub')
    import numpy as np
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from torch.nn.functional import softmax
    from sklearn.linear_model import Ridge
    import matplotlib.pyplot as plt

# HF token resolution (Colab userdata, then env)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    raise RuntimeError('Llama 3.1 8B is gated. Set HF_TOKEN as an env var or Colab secret.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'meta-llama/Llama-3.1-8B'
MAX_NEW_TOKENS = 512

BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.5, 'axes.labelcolor': FG,
    'axes.titlecolor': FG, 'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.6,
})


RECOMPUTE = True

OUT_DIR = Path('./out/multilayer_ablation').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

PERIODS = [2, 5, 10, 20, 50, 100]
LAYERS_TO_TEST = [15, 16, 17, 18, 19, 20, 21, 22]
RIDGE_ALPHA = 100.0
SEED = 42
N_ADD = 2000

DATA_FALLBACK_P1 = Path('../data/multilayer_ablation_phase1.json')
DATA_FALLBACK_P2 = Path('../data/multilayer_ablation_phase2.json')


In [ ]:
# Cell 2 — load Llama (only if RECOMPUTE)
if RECOMPUTE:
    # Load Llama-3.1-8B (gated; needs HF_TOKEN)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tok.padding_side = 'left'
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    
    t0 = time.time()
    MODEL = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        attn_implementation='sdpa', device_map='auto', token=HF_TOKEN,
    ).eval()
    for p in MODEL.parameters():
        p.requires_grad_(False)
    print(f'Loaded {MODEL_NAME} in {time.time()-t0:.1f}s on {DEVICE}')
    

else:
    print('RECOMPUTE = False; will replot from bundled JSON.')


In [ ]:
# Cell 3 — capture residuals at each layer for 2000 random addition prompts
if RECOMPUTE:
    rng_d = np.random.default_rng(SEED)
    add_pairs = [(int(rng_d.integers(0, 100)), int(rng_d.integers(0, 100))) for _ in range(N_ADD)]
    add_prompts = [f'{a}+{b}=' for a, b in add_pairs]
    add_sums = np.array([a + b for a, b in add_pairs])

    @torch.no_grad()
    def capture_layers_batch(prompts, layer_ids, batch_size=16):
        per_layer = {L: [] for L in layer_ids}
        for i in range(0, len(prompts), batch_size):
            ids = tok(prompts[i:i + batch_size], return_tensors='pt', padding=True).to(DEVICE)
            captured = {L: None for L in layer_ids}
            handles = []
            def make_hk(L):
                def hk(m, inp, out):
                    h = out[0] if isinstance(out, tuple) else out
                    captured[L] = h[:, -1, :].detach().float().cpu()
                return hk
            for L in layer_ids:
                handles.append(MODEL.model.layers[L].register_forward_hook(make_hk(L)))
            try: MODEL(**ids)
            finally:
                for h in handles: h.remove()
            for L in layer_ids: per_layer[L].append(captured[L])
        return {L: torch.cat(per_layer[L], dim=0).numpy() for L in layer_ids}

    print(f'Capturing {N_ADD} addition residuals at layers {LAYERS_TO_TEST}...')
    t0 = time.time()
    R_per_L = capture_layers_batch(add_prompts, LAYERS_TO_TEST)
    print(f'  done in {time.time()-t0:.1f}s')


In [ ]:
# Cell 4 — train Fourier probes at each layer + orthonormalize to a 12-dim U
if RECOMPUTE:
    def train_probes(R, sums, periods=PERIODS, alpha=RIDGE_ALPHA, test_frac=0.2, seed=SEED):
        n = len(sums)
        rng = np.random.RandomState(seed)
        idx = rng.permutation(n)
        n_train = int(n * (1 - test_frac))
        tr, te = idx[:n_train], idx[n_train:]
        W_rows = []; R2 = {}
        for T in periods:
            sin_t = np.sin(2 * np.pi * sums / T); cos_t = np.cos(2 * np.pi * sums / T)
            sp = Ridge(alpha=alpha).fit(R[tr], sin_t[tr])
            cp = Ridge(alpha=alpha).fit(R[tr], cos_t[tr])
            sin_pred = sp.predict(R[te]); cos_pred = cp.predict(R[te])
            joint_var = sin_t[te].var() + cos_t[te].var()
            joint_res = ((sin_pred - sin_t[te]) ** 2).mean() + ((cos_pred - cos_t[te]) ** 2).mean()
            R2[T] = float(1 - joint_res / max(1e-12, joint_var))
            W_rows.append(sp.coef_.astype(np.float32))
            W_rows.append(cp.coef_.astype(np.float32))
        return np.stack(W_rows, axis=0), R2

    def orthonormalize(W):
        if W.shape[0] == 0: return W
        Q, _ = np.linalg.qr(W.T); return Q.T

    U_per_L = {}; R2_per_L = {}
    print('Training 12-dim Fourier probe basis at each layer:')
    for L in LAYERS_TO_TEST:
        W, R2 = train_probes(R_per_L[L], add_sums)
        U_per_L[L] = orthonormalize(W)
        R2_per_L[L] = R2
        print(f'  L={L:>2d}  max R²={max(R2.values()):.3f}  mean R²={np.mean(list(R2.values())):.3f}  '
              f'U shape={U_per_L[L].shape}')


In [ ]:
# Cell 5 — multi-layer ablation hook + task helpers
if RECOMPUTE:
    class MultiLayerAblation:
        def __init__(self, layer_to_U):
            self.layer_to_U = layer_to_U; self._handles = []
        def __enter__(self):
            for L, U_np in self.layer_to_U.items():
                if U_np is None or U_np.shape[0] == 0: continue
                U_dev = torch.from_numpy(U_np).to(DEVICE).float()
                def make_hook(U):
                    def hook(m, inp, out):
                        h = out[0] if isinstance(out, tuple) else out
                        last = h[:, -1, :].float()
                        coords = last @ U.T
                        h[:, -1, :] = (last - coords @ U).to(h.dtype)
                        return (h,) + out[1:] if isinstance(out, tuple) else h
                    return hook
                self._handles.append(MODEL.model.layers[L].register_forward_hook(make_hook(U_dev)))
            return self
        def __exit__(self, *a):
            for h in self._handles: h.remove()

    DAYS = ['Saturday','Sunday','Monday','Tuesday','Wednesday','Thursday','Friday']
    OFFSETS_W = ['one','two','three','four','five','six','seven','eight','nine','ten',
                 'eleven','twelve','thirteen','fourteen']
    MONTHS = ['January','February','March','April','May','June','July','August',
              'September','October','November','December']
    OFFSETS_M = ['one','two','three','four','five','six','seven','eight','nine','ten','eleven','twelve']

    addition_subset = [{'prompt': f'{a}+{b}=', 'expected': str(a + b)}
                       for a in range(1, 13) for b in range(1, 13)]
    weekdays_subset = [{'prompt': f'Q: What day is {o} days after {d}?\nA:',
                        'expected': DAYS[(di + (oi + 1)) % 7]}
                       for di, d in enumerate(DAYS) for oi, o in enumerate(OFFSETS_W)]
    months_subset = [{'prompt': f'Q: What month is {o} months after {m}?\nA:',
                      'expected': MONTHS[((mi + 1) + (oi + 1) - 1) % 12]}
                     for mi, m in enumerate(MONTHS) for oi, o in enumerate(OFFSETS_M)]
    capital_subset = [{'prompt': f'The capital of {c} is', 'expected': cap} for c, cap in [
        ('France','Paris'),('Germany','Berlin'),('Italy','Rome'),('Spain','Madrid'),
        ('Portugal','Lisbon'),('Russia','Moscow'),('Poland','Warsaw'),('Sweden','Stockholm'),
        ('Norway','Oslo'),('Finland','Helsinki'),('Denmark','Copenhagen'),('Belgium','Brussels'),
        ('Switzerland','Bern'),('Austria','Vienna'),('Greece','Athens'),('Turkey','Ankara'),
        ('Egypt','Cairo'),('Israel','Jerusalem'),('Iran','Tehran'),('Iraq','Baghdad'),
        ('Japan','Tokyo'),('China','Beijing'),('India','Delhi'),('Thailand','Bangkok'),
        ('Vietnam','Hanoi'),('Indonesia','Jakarta'),('Australia','Canberra'),('Canada','Ottawa'),
        ('Chile','Santiago'),('Peru','Lima'),('Kenya','Nairobi'),('Morocco','Rabat'),
        ('Hungary','Budapest'),('Romania','Bucharest'),('Bulgaria','Sofia'),('Serbia','Belgrade'),
        ('Croatia','Zagreb'),('Czechia','Prague'),('Slovakia','Bratislava')
    ]]

    TASKS = [
        ('addition',        addition_subset, lambda e, p: p == e),
        ('weekdays',        weekdays_subset, lambda e, p: e.lower() in p.lower()),
        ('months',          months_subset,   lambda e, p: e.lower() in p.lower()),
        ('country_capital', capital_subset,  lambda e, p: e.lower() in p.lower()),
    ]
    for n, ps, _ in TASKS: print(f'  {n}: {len(ps)} prompts')

    @torch.no_grad()
    def measure_accuracy(layer_to_U):
        accs = {}
        for name, prompts, chk in TASKS:
            n_ok = 0
            for p in prompts:
                ids = tok(p['prompt'], return_tensors='pt').input_ids.to(DEVICE)
                if layer_to_U:
                    with MultiLayerAblation(layer_to_U):
                        pred_tok = MODEL(ids).logits[0, -1].argmax().item()
                else:
                    pred_tok = MODEL(ids).logits[0, -1].argmax().item()
                if chk(p['expected'], tok.decode(pred_tok).strip()):
                    n_ok += 1
            accs[name] = n_ok / max(1, len(prompts))
        return accs

    print('Helpers ready')


In [ ]:
# Cell 6 — Phase 1: single-layer ablation, baseline + each layer alone
if RECOMPUTE:
    phase1 = {}
    t0 = time.time()
    phase1['baseline'] = measure_accuracy({})
    print(f'baseline ({time.time()-t0:.0f}s): ' +
          ' '.join(f'{n}={a:.3f}' for n, a in phase1['baseline'].items()))
    for L in LAYERS_TO_TEST:
        t0 = time.time()
        accs = measure_accuracy({L: U_per_L[L]})
        phase1[f'L{L}'] = accs
        drops = {n: phase1['baseline'][n] - accs[n] for n in accs}
        print(f'L={L:>2d} ({time.time()-t0:.0f}s): ' +
              ' '.join(f'{n}={accs[n]:.3f} (Δ={drops[n]:+.3f})'
                       for n in ['addition','weekdays','months','country_capital']))
    with open(OUT_DIR / 'phase1.json', 'w') as f:
        json.dump(phase1, f, indent=2)
else:
    with open(DATA_FALLBACK_P1) as f:
        phase1 = json.load(f)
    print(f'Loaded phase1 from {DATA_FALLBACK_P1}')


In [ ]:
# Cell 7 — Phase 2: incremental joint multi-layer ablation
PHASE2_CONFIGS = [
    ('L18 only',          [18]),
    ('L17-19 (3 layers)', [17, 18, 19]),
    ('L16-20 (5 layers)', [16, 17, 18, 19, 20]),
    ('L15-22 (8 layers)', [15, 16, 17, 18, 19, 20, 21, 22]),
]

if RECOMPUTE:
    phase2 = {'baseline': phase1['baseline']}
    for name, layers in PHASE2_CONFIGS:
        t0 = time.time()
        layer_to_U = {L: U_per_L[L] for L in layers}
        accs = measure_accuracy(layer_to_U)
        phase2[name] = {'layers': layers, 'n_layers': len(layers),
                        'dim_total': len(layers) * 12, 'accs': accs}
        drops = {n: phase1['baseline'][n] - accs[n] for n in accs}
        print(f'{name:>20s} ({time.time()-t0:.0f}s): ' +
              ' '.join(f'{n}={accs[n]:.3f} (Δ={drops[n]:+.3f})'
                       for n in ['addition','weekdays','months','country_capital']))
    with open(OUT_DIR / 'phase2.json', 'w') as f:
        json.dump(phase2, f, indent=2)
else:
    with open(DATA_FALLBACK_P2) as f:
        phase2 = json.load(f)
    print(f'Loaded phase2 from {DATA_FALLBACK_P2}')


In [ ]:
# Cell 8 — Figure 2: Phase 1 + Phase 2 in one two-panel plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
task_colors = {'addition': '#cc4444', 'weekdays': '#4477cc',
               'months': '#44aa44', 'country_capital': '#888888'}
base = phase1['baseline']

# Left: Phase 1 single-layer Δaccuracy
for tname in ['addition', 'weekdays', 'months', 'country_capital']:
    drops = [base[tname] - phase1[f'L{L}'][tname] for L in LAYERS_TO_TEST]
    axes[0].plot(LAYERS_TO_TEST, drops, '-o', color=task_colors[tname], label=tname)
axes[0].axhline(0, ls='-', color='black', lw=0.5)
axes[0].set_xlabel('Single layer ablated (12-dim Fourier subspace)')
axes[0].set_ylabel('Accuracy drop (baseline − ablated)')
axes[0].set_title('Phase 1: per-layer single-ablation effect')
axes[0].legend()

# Right: Phase 2 joint ablation curve
configs = [c for c, _ in PHASE2_CONFIGS]
n_layers_list = [phase2[c]['n_layers'] for c in configs]
for tname in ['addition', 'weekdays', 'months', 'country_capital']:
    accs = [phase2[c]['accs'][tname] for c in configs]
    axes[1].plot([0] + n_layers_list, [base[tname]] + accs,
                 '-o', color=task_colors[tname], label=tname)
    chance = {'addition': 1/24, 'weekdays': 1/7, 'months': 1/12, 'country_capital': 0.0}[tname]
    axes[1].axhline(chance, ls=':', color=task_colors[tname], alpha=0.4)
axes[1].set_xlabel('Number of layers ablated (joint)')
axes[1].set_ylabel('Task accuracy')
axes[1].set_title('Phase 2: joint multi-layer ablation\n(dotted = chance per task)')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(OUT_DIR / 'plot.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {OUT_DIR/"plot.png"}')

print('\nPhase 1 summary (single-layer drops):')
print(f'  {"L":>3s}  ' + '  '.join(f'{n[:10]:>10s}' for n in ['addition','weekdays','months','country_capital']))
for L in LAYERS_TO_TEST:
    row = f'  {L:>3d}  ' + '  '.join(
        f'{base[n] - phase1[f"L{L}"][n]:>+10.3f}'
        for n in ['addition','weekdays','months','country_capital'])
    print(row)


## Expected results

**Phase 1.**
- Layers 15-17 show ~0 drop on every task. The calculator does not exist below L=18.
- Layers 18, 19, 20 each drop addition accuracy by 45-49pp on their own; weekdays drop ~35pp; months ~25pp. Each layer carries an independently causal subspace.
- Layers 21, 22 still drop counting noticeably (~20-30pp) but less than 18-20.
- The country-capital control sits at zero drop everywhere — confirms the ablation is counting-specific rather than a generic disturbance.

**Phase 2.**
- Joint ablation across all 8 layers (96 total subspace dimensions out of 4096) does not push counting to chance. Addition stays above 50%, weekdays above 25%. There is counting capability beyond the reach of any Fourier-style linear probe.
- Country-capital stays unaffected — the same hook applied to 8 layers does not break the model in general.

This panel matches `fig2_multilayer_ablation.png` in the longread. The next notebook (`manifold_walking.ipynb`) inspects what survives — and how to control it.
